# Lecture Bot

Ask questions about the documents in `docs/` (placeholders for now: syllabus and module
description). The switch below decides how the documents reach the model. You can flip it
between questions:

- **Context stuffing:** every question is sent together with all documents. Misses nothing,
  but everything must fit into the context window.
- **RAG** (Retrieval-Augmented Generation): a search picks the document chunks most similar
  to the question, and only those are sent. Scales to large collections, but the search can
  miss the right passage.

In [ ]:
import subprocess
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import ollama

from prompt_view import show_prompt

mode = widgets.ToggleButtons(options=[("Context stuffing", "stuffing"), ("RAG", "rag")],
                             value="stuffing")
mode

## 1. Load the documents

In [ ]:
def pdf_to_text(path):
    return subprocess.run(["pdftotext", "-layout", str(path), "-"],
                          capture_output=True, text=True, check=True).stdout

docs = {p.name: pdf_to_text(p) for p in sorted(Path("docs").glob("*.pdf"))}

for name, text in docs.items():
    print(f"{name}: ~{len(text) // 4} tokens")

<details>
<summary><b>What happens here</b></summary>

- Models read text, not PDFs. `pdftotext` extracts the text; images and layout are lost.
- A *token* is a word piece, about 4 characters. Characters divided by 4 is an estimate;
  the exact count appears in step 4.
</details>

## 2. Prepare the search (RAG only)

In [ ]:
def chunk(text, size=150, overlap=30):
    words = text.split()
    return [" ".join(words[i:i + size])
            for i in range(0, max(len(words) - overlap, 1), size - overlap)]

def embed(texts):
    return np.array(ollama.embed(model="nomic-embed-text", input=texts).embeddings)

chunks = [(name, c) for name, text in docs.items() for c in chunk(text)]
vectors = embed(["search_document: " + c for _, c in chunks])

def retrieve(question, k=3):
    q = embed(["search_query: " + question])[0]
    scores = vectors @ q / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))
    return [(*chunks[i], scores[i]) for i in scores.argsort()[::-1][:k]]

print(f"{len(chunks)} chunks embedded")

<details>
<summary><b>How the search works</b></summary>

1. `chunk()` splits each document into 150-word pieces. Neighbours overlap by 30 words, so a
   sentence cut at one border is whole in the next chunk.
2. `embed()` turns each chunk into a vector (a list of numbers) with the embedding model
   `nomic-embed-text`. Similar meaning gives similar vectors. The `search_document:` and
   `search_query:` prefixes are specific to this model, which was trained with them.
3. `retrieve()` embeds the question and scores every chunk by *cosine similarity*: 1 means
   same direction (similar meaning), 0 means unrelated. The best `k=3` chunks win.

150 words, 30 words overlap and `k=3` are common starting values, not tuned for these
documents.
</details>

## 3. Ask

In [ ]:
INSTRUCTIONS = """You are a teaching assistant for the module "Agentic AI".
Answer only from the lecture material below. If the material does not cover the
question, say so. Name the document you used. Answer in the language of the question."""

history = []

def ask(question):
    global last
    if mode.value == "stuffing":
        sources = [(name, text, None) for name, text in docs.items()]
    else:
        sources = retrieve(question)
    context = "\n\n".join(f"=== {name} ===\n{text}" for name, text, _ in sources)
    messages = ([{"role": "system", "content": INSTRUCTIONS + "\n\n" + context}]
                + history + [{"role": "user", "content": question}])

    response = ollama.chat(model="qwen2.5:7b", messages=messages,
                           options={"temperature": 0})
    answer = response.message.content

    history.extend([messages[-1], {"role": "assistant", "content": answer}])
    last = {"mode": mode.value, "model": "qwen2.5:7b", "sources": sources,
            "messages": messages, "answer": answer, "tokens": response.prompt_eval_count}
    print(answer)

<details>
<summary><b>Prompt and memory</b></summary>

- A call is a list of messages. The **system** message holds the rules plus the context and
  is rebuilt for every question; **user** and **assistant** messages are the conversation.
- The instructions each have a job:
  - *Answer only from the material*: prevents *hallucination*, confidently invented answers.
  - *If not covered, say so*: gives the model a permitted way out.
  - *Name the document*: makes the answer checkable.
- The model has **no memory**. `history` resends all earlier questions and answers with every
  call, so each call gets longer. `history.clear()` starts over.
- In RAG mode, a follow-up such as "and after that?" can retrieve poorly: the search sees
  only that one question.
- `temperature: 0` always picks the most likely token, so answers are reproducible.
- There is no context window setting: Ollama uses the model's maximum, 32,768 tokens for
  `qwen2.5:7b`. Everything sent must fit into it.
</details>

In [ ]:
ask("Wann wird MCP behandelt und worum geht es dabei?")

In [ ]:
ask("Und in welcher Woche kommen danach die Reasoning Models?")

In [ ]:
ask("Wie heißt der Hund des Professors?")

<details>
<summary><b>What the three questions test</b></summary>

1. A lookup: the answer is in the syllabus.
2. A follow-up: only works if the history is sent along.
3. Not in the material: the bot should say so, not invent a name.
</details>

In [ ]:
# Your questions:

## 4. What the model received

Replays your last question: every block of text the model read, in order. Click a block to
read it in full. Run the cell again to replay.

In [ ]:
show_prompt(last)

<details>
<summary><b>Which mode, and is this state of the art?</b></summary>

- **Documents fit into the window:** use stuffing. For small collections it is the
  recommended default; Anthropic, for example, advises skipping RAG below about 200,000
  tokens.
- **Documents do not fit:** use RAG. Production systems add keyword search (BM25), a
  reranking model, and tune chunk size and `k` by measuring answer quality.
- **Agentic RAG:** the model gets search as a tool and decides itself what to look up (the
  loop from Lab 02 with the tools from Lab 03). Lab 09 covers RAG in depth.
- `qwen2.5:7b` reads at most 32,768 tokens, roughly 50 pages. Commercial models accept
  hundreds of thousands.
- *Prompt injection* is not another word for stuffing: it names an attack where text inside
  a document overrides the instructions (Lab 13).
</details>